In [3]:
"""
Data Preprocessing Pipeline for ADAM-Sense Dataset
Prepares data for model training
"""

import pandas as pd
import numpy as np
from pathlib import Path
import pickle

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    # Sensors
    'sensor_columns': ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w'],
    'num_features': 6,
    
    # Activities (you can modify this list)
    'selected_activities': [
        'nail_biting',
        'knuckles_cracking', 
        'hand_tapping',
        'sitting'
    ],
    
    # Windowing
    'sampling_rate': 50,  # Hz
    'window_size': 128,  # samples (~2.56 seconds)
    'overlap': 0.5,  # 50%
    'step_size': 64,  # samples (window_size * overlap)
    
    # Train/test split by users
    'train_users': [3, 4, 5, 6, 7, 8, 9, 10],
    'test_users': [1, 2],
}

print("="*60)
print("DATA PREPROCESSING PIPELINE")
print("="*60)
print(f"\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# ============================================================================
# STEP 1: Load Dataset
# ============================================================================
print("\n" + "="*60)
print("STEP 1: Loading Dataset")
print("="*60)

data_path = Path('../data/raw/dataset.csv')
df = pd.read_csv(data_path)
print(f"✓ Loaded {len(df):,} samples")

# ============================================================================
# STEP 2: Filter Selected Activities
# ============================================================================
print("\n" + "="*60)
print("STEP 2: Filtering Activities")
print("="*60)

# Check which activities exist in the dataset
available_activities = df['Activity'].unique()
print(f"Available activities in dataset: {list(available_activities)}")

# Filter only selected activities
df_filtered = df[df['Activity'].isin(CONFIG['selected_activities'])]
print(f"\n✓ Filtered to {len(df_filtered):,} samples")

# Show distribution
print("\nSamples per activity:")
for activity in CONFIG['selected_activities']:
    count = len(df_filtered[df_filtered['Activity'] == activity])
    print(f"  {activity:20s}: {count:6,}")

# ============================================================================
# STEP 3: Create Activity Label Mapping
# ============================================================================
print("\n" + "="*60)
print("STEP 3: Creating Label Mapping")
print("="*60)

# Create label encoder
activity_to_label = {activity: idx for idx, activity in enumerate(CONFIG['selected_activities'])}
label_to_activity = {idx: activity for activity, idx in activity_to_label.items()}

print("Label mapping:")
for activity, label in activity_to_label.items():
    print(f"  {label}: {activity}")

# Add numeric label column
df_filtered['label'] = df_filtered['Activity'].map(activity_to_label)

# ============================================================================
# STEP 4: Extract Sensor Data by User
# ============================================================================
print("\n" + "="*60)
print("STEP 4: Extracting Sensor Data")
print("="*60)

def create_windows(data, labels, window_size, step_size):
    """
    Create sliding windows from continuous sensor data
    
    Args:
        data: numpy array of shape (samples, features)
        labels: numpy array of shape (samples,)
        window_size: number of samples per window
        step_size: step between windows (for overlap)
    
    Returns:
        X: windowed data of shape (num_windows, window_size, features)
        y: labels for each window of shape (num_windows,)
    """
    num_samples = data.shape[0]
    num_features = data.shape[1]
    
    windows = []
    window_labels = []
    
    # Slide window across data
    for start in range(0, num_samples - window_size + 1, step_size):
        end = start + window_size
        
        # Extract window
        window = data[start:end, :]
        
        # Get most common label in window (majority voting)
        window_label_values = labels[start:end]
        unique, counts = np.unique(window_label_values, return_counts=True)
        most_common_label = unique[np.argmax(counts)]
        
        # Only keep window if label is consistent (>80% same label)
        if np.max(counts) / len(window_label_values) > 0.8:
            windows.append(window)
            window_labels.append(most_common_label)
    
    return np.array(windows), np.array(window_labels)

# Process each user and activity separately
all_train_windows = []
all_train_labels = []
all_test_windows = []
all_test_labels = []

sensor_cols = CONFIG['sensor_columns']

for user in df_filtered['User'].unique():
    user_data = df_filtered[df_filtered['User'] == user]
    
    for activity in CONFIG['selected_activities']:
        activity_data = user_data[user_data['Activity'] == activity]
        
        if len(activity_data) < CONFIG['window_size']:
            continue  # Skip if not enough data
        
        # Extract sensor values and labels
        X = activity_data[sensor_cols].values
        y = activity_data['label'].values
        
        # Create windows
        X_windows, y_windows = create_windows(
            X, y, 
            CONFIG['window_size'], 
            CONFIG['step_size']
        )
        
        # Add to train or test set based on user
        if user in CONFIG['train_users']:
            all_train_windows.append(X_windows)
            all_train_labels.append(y_windows)
        elif user in CONFIG['test_users']:
            all_test_windows.append(X_windows)
            all_test_labels.append(y_windows)

# Concatenate all windows
X_train = np.concatenate(all_train_windows, axis=0)
y_train = np.concatenate(all_train_labels, axis=0)
X_test = np.concatenate(all_test_windows, axis=0)
y_test = np.concatenate(all_test_labels, axis=0)

print(f"✓ Created sliding windows")
print(f"  Training set: {X_train.shape[0]:,} windows")
print(f"  Test set: {X_test.shape[0]:,} windows")
print(f"  Window shape: {X_train.shape[1:]} (samples, features)")

# ============================================================================
# STEP 5: Normalize Data
# ============================================================================
print("\n" + "="*60)
print("STEP 5: Normalizing Data")
print("="*60)

# Calculate normalization parameters from TRAINING data only
mean = X_train.mean(axis=(0, 1))  # Mean across all windows and time steps
std = X_train.std(axis=(0, 1))    # Std across all windows and time steps

print("Normalization parameters (per feature):")
print("Mean:", mean)
print("Std:", std)

# Apply normalization
X_train_normalized = (X_train - mean) / std
X_test_normalized = (X_test - mean) / std

print(f"\n✓ Data normalized")
print(f"  Train range: [{X_train_normalized.min():.2f}, {X_train_normalized.max():.2f}]")
print(f"  Test range: [{X_test_normalized.min():.2f}, {X_test_normalized.max():.2f}]")

# ============================================================================
# STEP 6: Verify Class Distribution
# ============================================================================
print("\n" + "="*60)
print("STEP 6: Class Distribution")
print("="*60)

print("Training set:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for label, count in zip(unique_train, counts_train):
    activity_name = label_to_activity[label]
    percentage = (count / len(y_train)) * 100
    print(f"  {label}: {activity_name:20s} - {count:6,} ({percentage:5.2f}%)")

print("\nTest set:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for label, count in zip(unique_test, counts_test):
    activity_name = label_to_activity[label]
    percentage = (count / len(y_test)) * 100
    print(f"  {label}: {activity_name:20s} - {count:6,} ({percentage:5.2f}%)")

# ============================================================================
# STEP 7: Save Preprocessed Data
# ============================================================================
print("\n" + "="*60)
print("STEP 7: Saving Preprocessed Data")
print("="*60)

# Create output directory
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save as numpy arrays
np.save(output_dir / 'X_train.npy', X_train_normalized)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'X_test.npy', X_test_normalized)
np.save(output_dir / 'y_test.npy', y_test)

# Save normalization parameters and config
preprocessing_info = {
    'config': CONFIG,
    'mean': mean,
    'std': std,
    'activity_to_label': activity_to_label,
    'label_to_activity': label_to_activity,
    'train_shape': X_train_normalized.shape,
    'test_shape': X_test_normalized.shape,
}

with open(output_dir / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print(f"✓ Saved to {output_dir}/")
print(f"  - X_train.npy: {X_train_normalized.shape}")
print(f"  - y_train.npy: {y_train.shape}")
print(f"  - X_test.npy: {X_test_normalized.shape}")
print(f"  - y_test.npy: {y_test.shape}")
print(f"  - preprocessing_info.pkl")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)

print(f"""
Summary:
  ✓ Selected {len(CONFIG['selected_activities'])} activities
  ✓ Training users: {CONFIG['train_users']}
  ✓ Test users: {CONFIG['test_users']}
  ✓ Window size: {CONFIG['window_size']} samples (~{CONFIG['window_size']/CONFIG['sampling_rate']:.2f} seconds)
  ✓ Overlap: {CONFIG['overlap']*100:.0f}%
  ✓ Features: {CONFIG['num_features']} (accel + gyro)
  
  Training set: {X_train_normalized.shape[0]:,} windows
  Test set: {X_test_normalized.shape[0]:,} windows
  
Next step: Model Training
  → Run: 03_model_training.ipynb
""")

DATA PREPROCESSING PIPELINE

Configuration:
  sensor_columns: ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w']
  num_features: 6
  selected_activities: ['nail_biting', 'knuckles_cracking', 'hand_tapping', 'sitting']
  sampling_rate: 50
  window_size: 128
  overlap: 0.5
  step_size: 64
  train_users: [3, 4, 5, 6, 7, 8, 9, 10]
  test_users: [1, 2]

STEP 1: Loading Dataset
✓ Loaded 709,582 samples

STEP 2: Filtering Activities
Available activities in dataset: ['knuckles_cracking', 'hand_scratching', 'hair_pulling', 'smoking', 'ear_rubbing', 'forehead_rubbing', 'nape_rubbing', 'sitting', 'standing', 'nail_biting', 'hand_tapping']

✓ Filtered to 254,171 samples

Samples per activity:
  nail_biting         : 63,457
  knuckles_cracking   : 67,498
  hand_tapping        : 63,220
  sitting             : 59,996

STEP 3: Creating Label Mapping
Label mapping:
  0: nail_biting
  1: knuckles_cracking
  2: hand_tapping
  3: sitting

STEP 4: Extracting Sensor Data
✓ Created sliding windows
  Training s

/var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/ipykernel_30406/2611324903.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['label'] = df_filtered['Activity'].map(activity_to_label)
